# Optimized Search V3 — Composite Confidence Score

**V2'den ogrendiklerimiz:**
- pTM=0.6 cutoff N=214 icin ulasilamaz -> alpha death spiral
- warmup>=5 felaket (Rg patlamasi)
- alpha_decay=0.8 en buyuk etki (6/30 accept vs 2/30 baseline)
- stall=8 gereksiz GPU hesaplamayi onluyor

**V3 stratejisi:** Tek tek metrik cutoff'lari yerine **composite score**.
Her metrik 0-1'e normalize edilir, agirlikli toplanir, tek threshold ile karar verilir.

**Tahmini sure:** ~2-3 saat (V2'nin 9 saatinden 3x hizli)

**Calistirma:** Cell 1 (setup) -> Cell 2 (config) -> Cell 3 (OF3) -> Cell 4+ (deneyler)

## 1. Environment Setup (bir kere calistir)

In [ ]:
import os, sys, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch pandas

sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print('Environment setup complete.')

## 2. Config

In [ ]:
# ════════════════════════════════════════════════════
#  CONFIG — V3
# ════════════════════════════════════════════════════

# ── PDB ────────────────────────────────────────────
INITIAL_PDB = '1AKE'
TARGET_PDB  = '4AKE'
CHAIN_ID    = 'A'

# ── Pipeline (V2'den ogrenilenlerle optimize) ──────
N_STEPS              = 20     # 30->20: son 10 step zaten stall
COMBINATION_STRATEGY = 'autostop'
Z_MIXING_ALPHA       = 0.7
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# ── OF3 ────────────────────────────────────────────
USE_MSA_SERVER       = True
USE_TEMPLATES        = False
NUM_ROLLOUT_STEPS    = None
NUM_DIFFUSION_SAMPLES = 1

# ── V2'den sabit parametreler (degistirmeyecegiz) ──
ALPHA_DECAY          = 0.8     # V2 best
MAX_STALL            = 8       # V2 best
CONFIDENCE_RG_MAX    = 2.0     # V3: sikistirildi — Rg>2.0 hard reject
CONFIDENCE_RG_MIN    = 0.3
CONFIDENCE_CLASH_REJECT = True

# ── Fallback ───────────────────────────────────────
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# ── Autostop ───────────────────────────────────────
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# ── Converter ─────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

print('Config ready.')

## 3. Converter + PDB + OF3 yukleme

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import torch
import time
import copy
import pandas as pd

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.composite_confidence import (
    CompositeWeights, compute_composite, composite_score_from_step,
    WEIGHT_PRESETS, THRESHOLD_PRESETS,
    normalize_ptm, normalize_plddt, normalize_pae, normalize_rg, normalize_contact_recon,
)

# ── Checkpoint ──────────────────────────────────────
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded from {CHECKPOINT_PATH}')

# ── PDB ──────────────────────────────────────────────
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

def fetch_ca(pdb_id, chain_id):
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence

initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
target_ca, _ = fetch_ca(TARGET_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'Initial: {INITIAL_PDB} chain {CHAIN_ID}, N={N}')
print(f'Target:  {TARGET_PDB} chain {CHAIN_ID}, N={target_ca.shape[0]}')
print(f'RMSD to target: {compute_rmsd(initial_ca, target_ca):.2f} A')
print(f'TM-score: {tm_score(initial_ca, target_ca):.4f}')

# ── OF3 ──────────────────────────────────────────────
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)
_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{
                'molecule_type': 'protein',
                'chain_ids': [CHAIN_ID],
                'sequence': sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion
diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')

from src.autostop_adapter import StructureContext
_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print(f'StructureContext: N={structure_ctx.struct.N}')
print('OF3 ready.')

## 4. Composite Score Runner

In [ ]:
# ════════════════════════════════════════════════════
#  COMPOSITE SCORE RUNNER
#  V2'nin _confidence_check AND logic'i yerine
#  weighted composite score kullanir.
# ════════════════════════════════════════════════════

def make_config(
    n_steps=N_STEPS,
    alpha=Z_MIXING_ALPHA,
    alpha_decay=ALPHA_DECAY,
    max_stall=MAX_STALL,
    # Composite mode: pTM cutoff cok dusuk tutulur ki
    # fallback cascade hala calisssin ama
    # accept/reject karari composite'e birakilsin
    ptm_cutoff=0.15,      # cok dusuk — neredeyse her sey L0/L1'den gecer
    plddt_cutoff=40.0,    # cok dusuk
    ranking_cutoff=0.10,  # cok dusuk
):
    """ModeDriveConfig with V2 learnings baked in.
    
    pTM/pLDDT/ranking cutoff'lari cok dusuk tutulur ki
    fallback ladder gereksiz yere L9'a cascade yapmasin.
    Gercek accept/reject composite score ile yapilir.
    """
    return ModeDriveConfig(
        n_steps=n_steps,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=alpha,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        # Dusuk cutoff'lar — fallback cascade'i hafifletir
        confidence_ptm_cutoff=ptm_cutoff,
        confidence_plddt_cutoff=plddt_cutoff,
        confidence_ranking_cutoff=ranking_cutoff,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=CHAIN_ID,
        # V2 learnings
        rejected_alpha_decay=alpha_decay,
        max_consecutive_rejected=max_stall,
        confidence_warmup_steps=0,  # warmup kapalı
    )


def run_composite(
    weights: CompositeWeights,
    threshold: float,
    label: str = '',
    n_steps: int = N_STEPS,
    alpha: float = Z_MIXING_ALPHA,
    alpha_decay: float = ALPHA_DECAY,
    max_stall: int = MAX_STALL,
    early_abort_steps: int = 5,   # ilk N step'te 0 accept -> abort
    verbose: bool = True,
) -> dict:
    """Run pipeline with composite score accept/reject.
    
    Pipeline'in kendi _confidence_check'i dusuk cutoff'larla
    neredeyse her zaman PASS verir. Gercek accept/reject
    composite score'a gore yapilir.
    """
    cfg = make_config(
        n_steps=n_steps, alpha=alpha,
        alpha_decay=alpha_decay, max_stall=max_stall,
    )
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    print(f'\n{"="*70}')
    print(f'  {label}')
    print(f'  weights: {weights.as_dict()}')
    print(f'  threshold={threshold:.2f}  alpha={alpha}  decay={alpha_decay}  stall={max_stall}')
    print(f'{"="*70}')

    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = alpha
    current_alpha = alpha
    consecutive_rejected = 0

    step_metrics = []
    t0 = time.time()

    for step_idx in range(n_steps):
        # Pipeline step — dusuk cutoff ile cogu L0/L1'den gecer
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca,
            step_idx=step_idx,
        )

        # Composite score hesapla
        comp_score, comp_detail = composite_score_from_step(sr, weights)

        # Accept/reject: composite score >= threshold
        # Ama Rg ve clash hard-reject olarak kalir
        hard_reject = False
        reject_reason = ''
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            hard_reject = True
            reject_reason = f'Rg={sr.rg_ratio:.1f}>{CONFIDENCE_RG_MAX}'
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            hard_reject = True
            reject_reason = 'clash'

        if hard_reject:
            accepted = False
        else:
            accepted = comp_score >= threshold
            if not accepted:
                reject_reason = f'comp={comp_score:.3f}<{threshold}'

        # Target metrikleri
        rmsd_tgt = compute_rmsd(sr.new_ca, target_ca)
        tm_tgt = tm_score(sr.new_ca, target_ca)

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'comp_score': comp_score,
            'rmsd_init': sr.rmsd,
            'rmsd_tgt': rmsd_tgt,
            'tm_tgt': tm_tgt,
            'ptm': sr.ptm,
            'plddt_mean': float(sr.plddt.mean()) if sr.plddt is not None else None,
            'ranking': sr.ranking_score,
            'mean_pae': sr.mean_pae,
            'rg_ratio': sr.rg_ratio,
            'contact_recon': sr.contact_recon,
            'contact_of3': sr.contact_of3,
            'has_clash': sr.has_clash,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
            'reject_reason': reject_reason,
            **{f'd_{k}': v for k, v in comp_detail.items()},
        }
        step_metrics.append(m)

        if verbose:
            tag = 'ACCEPT' if accepted else 'REJECT'
            ptm_s = f'{sr.ptm:.3f}' if sr.ptm is not None else '-'
            pae_s = f'{sr.mean_pae:.1f}' if sr.mean_pae is not None else '-'
            rg_s = f'{sr.rg_ratio:.2f}' if sr.rg_ratio is not None else '-'
            print(
                f'  [{step_idx+1:>2}/{n_steps}] {tag:<6}  '
                f'comp={comp_score:.3f}  '
                f'pTM={ptm_s}  mPAE={pae_s}  Rg={rg_s}  '
                f'RMSD_tgt={rmsd_tgt:.2f}  TM={tm_tgt:.3f}  '
                f'FB={sr.fallback_level}  a={current_alpha:.3f}'
                f'  {reject_reason}' if not accepted else
                f'  [{step_idx+1:>2}/{n_steps}] {tag:<6}  '
                f'comp={comp_score:.3f}  '
                f'pTM={ptm_s}  mPAE={pae_s}  Rg={rg_s}  '
                f'RMSD_tgt={rmsd_tgt:.2f}  TM={tm_tgt:.3f}  '
                f'FB={sr.fallback_level}  a={current_alpha:.3f}'
            )

        # Update state
        if accepted:
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha  # restore
        else:
            consecutive_rejected += 1
            if alpha_decay < 1.0:
                current_alpha = max(0.02, current_alpha * alpha_decay)
            if max_stall > 0 and consecutive_rejected >= max_stall:
                if verbose:
                    print(f'  STALL: {consecutive_rejected} consecutive rejected')
                break

        # Early abort: ilk N step'te 0 accept -> bu config ise yaramaz
        if step_idx + 1 == early_abort_steps:
            n_accepted = sum(1 for m in step_metrics if m['accepted'])
            if n_accepted == 0:
                if verbose:
                    print(f'  EARLY ABORT: 0/{early_abort_steps} accepted in first steps')
                break

    wall = time.time() - t0
    total_steps = len(step_metrics)
    n_accepted = sum(1 for m in step_metrics if m['accepted'])
    n_rejected = total_steps - n_accepted

    # Best accepted step (by TM-score)
    accepted_metrics = [m for m in step_metrics if m['accepted']]
    if accepted_metrics:
        best_step = max(accepted_metrics, key=lambda m: m['tm_tgt'])
        best_tm = best_step['tm_tgt']
        best_rmsd = best_step['rmsd_tgt']
    else:
        best_tm = tm_score(initial_ca, target_ca)
        best_rmsd = compute_rmsd(initial_ca, target_ca)

    # Last accepted state metrics
    last_tm = step_metrics[-1]['tm_tgt'] if step_metrics else 0
    last_rmsd = step_metrics[-1]['rmsd_tgt'] if step_metrics else 0

    result = {
        'label': label,
        'weights_name': label.split('_')[0] if '_' in label else label,
        'threshold': threshold,
        'alpha': alpha,
        'alpha_decay': alpha_decay,
        'total_steps': total_steps,
        'accepted': n_accepted,
        'rejected': n_rejected,
        'accept_rate': n_accepted / max(1, total_steps),
        'best_tm': best_tm,
        'best_rmsd': best_rmsd,
        'last_tm': last_tm,
        'last_rmsd': last_rmsd,
        'wall_s': wall,
        'step_metrics': step_metrics,
    }

    if verbose:
        print(f'\n  => {n_accepted}/{total_steps} accepted ({n_accepted/max(1,total_steps)*100:.0f}%)  '
              f'best_TM={best_tm:.4f}  best_RMSD={best_rmsd:.2f}  wall={wall:.0f}s')

    return result


ALL_RESULTS = {}
print('Runner ready.')

## 5. Phase 1: Agirlik Seti Taramasi

4 agirlik seti x 3 threshold = 12 run

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 1: WEIGHT SET GRID
#  4 agirlik seti x 3 threshold = 12 run
# ════════════════════════════════════════════════════

ALL_RESULTS['phase1_weights'] = []

for wname, weights in WEIGHT_PRESETS.items():
    for thr in THRESHOLD_PRESETS:
        label = f'{wname}_t{thr:.2f}'
        r = run_composite(
            weights=weights,
            threshold=thr,
            label=label,
        )
        ALL_RESULTS['phase1_weights'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase1_weights']:
    rows.append({
        'label': r['label'],
        'threshold': r['threshold'],
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'last_TM': f"{r['last_tm']:.4f}",
        'wall_s': f"{r['wall_s']:.0f}",
    })
df1 = pd.DataFrame(rows)
print('\n' + '='*80)
print('PHASE 1 RESULTS')
print('='*80)
print(df1.to_string(index=False))

# En iyi config
best_p1 = max(ALL_RESULTS['phase1_weights'], key=lambda r: (r['accepted'], r['best_tm']))
print(f'\nBest Phase 1: {best_p1["label"]} — '
      f'{best_p1["accepted"]}/{best_p1["total_steps"]} accepted, '
      f'TM={best_p1["best_tm"]:.4f}')

## 6. Phase 2: Alpha Schedule Karsilastirmasi

En iyi agirlik seti ile 4 farkli alpha stratejisi

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 2: ALPHA SCHEDULE
#  Phase 1'in en iyi weight+threshold'u ile
#  farkli alpha stratejileri
# ════════════════════════════════════════════════════

# Phase 1 en iyi sonucunu al
best_p1 = max(ALL_RESULTS['phase1_weights'], key=lambda r: (r['accepted'], r['best_tm']))
best_wname = best_p1['label'].rsplit('_t', 1)[0]
best_thr = best_p1['threshold']
best_weights = WEIGHT_PRESETS[best_wname]

print(f'Phase 1 best: {best_wname} @ threshold={best_thr}')
print(f'Weights: {best_weights.as_dict()}')

alpha_configs = [
    {'alpha': 0.7, 'alpha_decay': 1.0,  'label': 'no_decay'},
    {'alpha': 0.7, 'alpha_decay': 0.9,  'label': 'slow_decay'},
    {'alpha': 0.7, 'alpha_decay': 0.8,  'label': 'medium_decay'},
    {'alpha': 0.7, 'alpha_decay': 0.7,  'label': 'fast_decay'},
    {'alpha': 0.5, 'alpha_decay': 0.85, 'label': 'low_start'},
    {'alpha': 0.3, 'alpha_decay': 1.0,  'label': 'conservative'},
]

ALL_RESULTS['phase2_alpha'] = []

for ac in alpha_configs:
    label = f'alpha_{ac["label"]}'
    r = run_composite(
        weights=best_weights,
        threshold=best_thr,
        label=label,
        alpha=ac['alpha'],
        alpha_decay=ac['alpha_decay'],
    )
    ALL_RESULTS['phase2_alpha'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase2_alpha']:
    rows.append({
        'label': r['label'],
        'alpha': r['alpha'],
        'decay': r['alpha_decay'],
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'wall_s': f"{r['wall_s']:.0f}",
    })
df2 = pd.DataFrame(rows)
print('\n' + '='*80)
print('PHASE 2 RESULTS')
print('='*80)
print(df2.to_string(index=False))

best_p2 = max(ALL_RESULTS['phase2_alpha'], key=lambda r: (r['accepted'], r['best_tm']))
print(f'\nBest Phase 2: {best_p2["label"]} — '
      f'{best_p2["accepted"]}/{best_p2["total_steps"]} accepted, '
      f'TM={best_p2["best_tm"]:.4f}')

## 7. Phase 3: pTM Cutoff Seviyesi (fallback cascade kontrolu)

Composite score main gate olsa bile, pipeline'in internal
fallback'i hala pTM cutoff'a bakar. Cok dusuk tutmak
fallback'i tamamen bypass eder (her sey L0'dan gecer),
biraz yukseltmek L1/L4'u aktif tutar.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 3: INTERNAL pTM CUTOFF LEVEL
#  Composite main gate olsa bile, fallback ladder
#  icindeki pTM cutoff hala etkili.
# ════════════════════════════════════════════════════

best_p2 = max(ALL_RESULTS['phase2_alpha'], key=lambda r: (r['accepted'], r['best_tm']))
best_alpha = best_p2['alpha']
best_decay = best_p2['alpha_decay']

ptm_levels = [
    {'ptm_cutoff': 0.10, 'ranking_cutoff': 0.05, 'label': 'ptm010'},  # neredeyse bypass
    {'ptm_cutoff': 0.20, 'ranking_cutoff': 0.10, 'label': 'ptm020'},
    {'ptm_cutoff': 0.35, 'ranking_cutoff': 0.20, 'label': 'ptm035'},
    {'ptm_cutoff': 0.50, 'ranking_cutoff': 0.30, 'label': 'ptm050'},  # V2'ye yakin
]

ALL_RESULTS['phase3_ptm'] = []

for pc in ptm_levels:
    label = f'fb_{pc["label"]}'
    cfg = make_config(
        alpha=best_alpha, alpha_decay=best_decay,
        ptm_cutoff=pc['ptm_cutoff'],
        ranking_cutoff=pc['ranking_cutoff'],
    )
    # Dogrudan run_composite cagiramayiz cunku ptm_cutoff custom.
    # make_config'i override edip cagirmamiz lazim.
    # Daha temizi: run_composite'a ptm_cutoff parametresi eklemek yerine
    # fonksiyonu biraz genisletelim.
    
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    print(f'\n{"="*70}')
    print(f'  {label} — pTM_cutoff={pc["ptm_cutoff"]} rank_cutoff={pc["ranking_cutoff"]}')
    print(f'{"="*70}')

    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    current_alpha = best_alpha
    consecutive_rejected = 0
    step_metrics = []
    t0 = time.time()

    for step_idx in range(N_STEPS):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            step_idx=step_idx, target_coords=target_ca,
        )
        comp_score, comp_detail = composite_score_from_step(sr, best_weights)

        hard_reject = False
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            hard_reject = True
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            hard_reject = True

        accepted = (not hard_reject) and (comp_score >= best_thr)
        rmsd_tgt = compute_rmsd(sr.new_ca, target_ca)
        tm_tgt = tm_score(sr.new_ca, target_ca)

        m = {
            'step': step_idx + 1, 'accepted': accepted,
            'comp_score': comp_score, 'rmsd_tgt': rmsd_tgt, 'tm_tgt': tm_tgt,
            'ptm': sr.ptm, 'mean_pae': sr.mean_pae, 'rg_ratio': sr.rg_ratio,
            'fallback_level': sr.fallback_level, 'alpha_used': current_alpha,
        }
        step_metrics.append(m)

        ptm_s = f'{sr.ptm:.3f}' if sr.ptm is not None else '-'
        tag = 'ACCEPT' if accepted else 'REJECT'
        print(f'  [{step_idx+1:>2}] {tag:<6} comp={comp_score:.3f} pTM={ptm_s} '
              f'TM={tm_tgt:.3f} FB={sr.fallback_level} a={current_alpha:.3f}')

        if accepted:
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = best_alpha
        else:
            consecutive_rejected += 1
            current_alpha = max(0.02, current_alpha * best_decay)
            if MAX_STALL > 0 and consecutive_rejected >= MAX_STALL:
                print(f'  STALL: {consecutive_rejected} consecutive rejected')
                break

    wall = time.time() - t0
    n_acc = sum(1 for m in step_metrics if m['accepted'])
    acc_steps = [m for m in step_metrics if m['accepted']]
    best_tm_val = max((m['tm_tgt'] for m in acc_steps), default=tm_score(initial_ca, target_ca))

    ALL_RESULTS['phase3_ptm'].append({
        'label': label, 'ptm_cutoff': pc['ptm_cutoff'],
        'total_steps': len(step_metrics), 'accepted': n_acc,
        'accept_rate': n_acc / max(1, len(step_metrics)),
        'best_tm': best_tm_val, 'wall_s': wall,
        'step_metrics': step_metrics,
    })
    print(f'  => {n_acc}/{len(step_metrics)} accepted, best_TM={best_tm_val:.4f}, wall={wall:.0f}s')

# Summary
rows = []
for r in ALL_RESULTS['phase3_ptm']:
    rows.append({
        'label': r['label'], 'ptm_cut': r['ptm_cutoff'],
        'steps': r['total_steps'], 'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}", 'wall': f"{r['wall_s']:.0f}s",
    })
print('\n' + '='*60)
print('PHASE 3 RESULTS')
print('='*60)
print(pd.DataFrame(rows).to_string(index=False))

## 8. Phase 4: Final Combined Run (30 step)

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE 4: FINAL COMBINED — 30 step ile en iyi config
# ════════════════════════════════════════════════════

# En iyi P1 + P2 + P3 sonuclarini birlestir
best_p1 = max(ALL_RESULTS['phase1_weights'], key=lambda r: (r['accepted'], r['best_tm']))
best_p2 = max(ALL_RESULTS['phase2_alpha'], key=lambda r: (r['accepted'], r['best_tm']))
best_p3 = max(ALL_RESULTS['phase3_ptm'], key=lambda r: (r['accepted'], r['best_tm']))

best_wname = best_p1['label'].rsplit('_t', 1)[0]
final_weights = WEIGHT_PRESETS[best_wname]
final_thr = best_p1['threshold']
final_alpha = best_p2['alpha']
final_decay = best_p2['alpha_decay']
final_ptm_cut = best_p3['ptm_cutoff']

print(f'Final config:')
print(f'  weights: {best_wname} -> {final_weights.as_dict()}')
print(f'  threshold: {final_thr}')
print(f'  alpha: {final_alpha}, decay: {final_decay}')
print(f'  internal pTM cutoff: {final_ptm_cut}')

# 30 step ile final run
r_final = run_composite(
    weights=final_weights,
    threshold=final_thr,
    label='FINAL_30step',
    n_steps=30,
    alpha=final_alpha,
    alpha_decay=final_decay,
    max_stall=12,  # 30 step icin biraz daha toleransli
    early_abort_steps=8,
)
ALL_RESULTS['phase4_final'] = [r_final]

print(f'\nFINAL: {r_final["accepted"]}/{r_final["total_steps"]} accepted '
      f'({r_final["accept_rate"]*100:.0f}%), '
      f'best_TM={r_final["best_tm"]:.4f}, '
      f'best_RMSD={r_final["best_rmsd"]:.2f}')

## 9. Analiz ve Grafikler

In [ ]:
# ════════════════════════════════════════════════════
#  ANALYSIS AND PLOTS
# ════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

def plot_phase_comparison(phase_results, title=''):
    """Accept rate vs best TM scatter."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # 1. Accept rate bar
    labels = [r['label'] for r in phase_results]
    accept_rates = [r['accept_rate'] * 100 for r in phase_results]
    colors = ['#2ecc71' if ar > 20 else '#e74c3c' for ar in accept_rates]
    axes[0].barh(labels, accept_rates, color=colors)
    axes[0].set_xlabel('Accept Rate (%)')
    axes[0].set_title('Accept Rate')
    axes[0].axvline(20, color='gray', linestyle='--', alpha=0.5, label='20% target')

    # 2. Best TM bar
    tms = [r['best_tm'] for r in phase_results]
    colors_tm = ['#3498db' if t > 0.6 else '#e67e22' for t in tms]
    axes[1].barh(labels, tms, color=colors_tm)
    axes[1].set_xlabel('Best TM-score')
    axes[1].set_title('Best TM-score')
    axes[1].axvline(0.6, color='gray', linestyle='--', alpha=0.5)

    # 3. Wall time
    walls = [r['wall_s'] for r in phase_results]
    axes[2].barh(labels, walls, color='#9b59b6')
    axes[2].set_xlabel('Wall time (s)')
    axes[2].set_title('Wall Time')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_step_trajectory(result, title=''):
    """Per-step composite score, TM, and accept/reject."""
    metrics = result['step_metrics']
    steps = [m['step'] for m in metrics]
    comp = [m['comp_score'] for m in metrics]
    tm = [m['tm_tgt'] for m in metrics]
    accepted = [m['accepted'] for m in metrics]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Composite score
    for i, (s, c, a) in enumerate(zip(steps, comp, accepted)):
        color = '#2ecc71' if a else '#e74c3c'
        ax1.scatter(s, c, color=color, s=80, zorder=5, edgecolors='black', linewidths=0.5)
    ax1.plot(steps, comp, 'k-', alpha=0.3)
    ax1.axhline(result.get('threshold', 0.45), color='orange', linestyle='--', label='threshold')
    ax1.set_ylabel('Composite Score')
    ax1.set_title(f'{title} — Composite Score (green=accept, red=reject)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # TM-score
    for i, (s, t, a) in enumerate(zip(steps, tm, accepted)):
        color = '#2ecc71' if a else '#e74c3c'
        ax2.scatter(s, t, color=color, s=80, zorder=5, edgecolors='black', linewidths=0.5)
    ax2.plot(steps, tm, 'k-', alpha=0.3)
    ax2.set_ylabel('TM-score vs target')
    ax2.set_xlabel('Step')
    ax2.set_title('TM-score vs Target')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_composite_breakdown(result, title=''):
    """Stacked bar of composite score components."""
    metrics = result['step_metrics']
    steps = [m['step'] for m in metrics]

    components = ['d_c_ptm', 'd_c_plddt', 'd_c_pae', 'd_c_rg', 'd_c_cr']
    comp_labels = ['pTM', 'pLDDT', 'PAE', 'Rg', 'ContactRecon']
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

    fig, ax = plt.subplots(figsize=(14, 5))
    bottom = np.zeros(len(steps))
    for comp_key, clabel, color in zip(components, comp_labels, colors):
        vals = np.array([m.get(comp_key, 0) for m in metrics])
        ax.bar(steps, vals, bottom=bottom, label=clabel, color=color, alpha=0.8)
        bottom += vals

    thr = result.get('threshold', 0.45)
    ax.axhline(thr, color='orange', linestyle='--', linewidth=2, label=f'threshold={thr}')
    ax.set_xlabel('Step')
    ax.set_ylabel('Composite Score')
    ax.set_title(f'{title} — Score Breakdown')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2, axis='y')
    plt.tight_layout()
    plt.show()


# Phase 1 comparison
if 'phase1_weights' in ALL_RESULTS:
    plot_phase_comparison(ALL_RESULTS['phase1_weights'], 'Phase 1: Weight Sets')

# Phase 2 comparison
if 'phase2_alpha' in ALL_RESULTS:
    plot_phase_comparison(ALL_RESULTS['phase2_alpha'], 'Phase 2: Alpha Schedules')

# Phase 3 comparison
if 'phase3_ptm' in ALL_RESULTS:
    plot_phase_comparison(ALL_RESULTS['phase3_ptm'], 'Phase 3: Internal pTM Cutoff')

# Final trajectory
if 'phase4_final' in ALL_RESULTS:
    r_final = ALL_RESULTS['phase4_final'][0]
    plot_step_trajectory(r_final, 'Final Run')
    plot_composite_breakdown(r_final, 'Final Run')

print('Analysis complete.')

## 10. V2 vs V3 Karsilastirmasi

In [ ]:
# ════════════════════════════════════════════════════
#  V2 BASELINE vs V3 COMPOSITE
#  V2 baseline: pTM=0.6, no decay, no stall
# ════════════════════════════════════════════════════

# V2 baseline run (ayni pipeline, eski cutoff'larla)
cfg_v2 = ModeDriveConfig(
    n_steps=20,
    combination_strategy=COMBINATION_STRATEGY,
    z_mixing_alpha=0.7,
    n_anm_modes=N_ANM_MODES,
    n_combinations=N_COMBINATIONS,
    max_combo_size=MAX_COMBO_SIZE,
    df=DF, df_min=DF_MIN, df_max=DF_MAX,
    anm_cutoff=ANM_CUTOFF,
    contact_r_cut=CONTACT_R_CUT,
    contact_tau=CONTACT_TAU,
    z_direction=Z_DIRECTION,
    num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
    confidence_ptm_cutoff=0.6,         # V2 original
    confidence_plddt_cutoff=70.0,       # V2 original
    confidence_ranking_cutoff=0.4,      # V2 original
    confidence_rg_max=2.5,
    confidence_rg_min=0.3,
    confidence_clash_reject=True,
    enable_confidence_fallback=True,
    fallback_extended_enabled=True,
    autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
    autostop_n_steps=AUTOSTOP_N_STEPS,
    autostop_back_off=AUTOSTOP_BACK_OFF,
    autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
    autostop_chain_id=CHAIN_ID,
    rejected_alpha_decay=1.0,          # no decay
    max_consecutive_rejected=0,        # no stall
)

pipe_v2 = ModeDrivePipeline(
    converter=converter, config=cfg_v2,
    diffusion_fn=diffusion_fn,
    structure_ctx=structure_ctx,
)

print('Running V2 baseline (20 steps)...')
t0 = time.time()
result_v2 = pipe_v2.run(
    initial_ca.clone(), zij_trunk.clone(),
    verbose=True, target_coords=target_ca,
)
wall_v2 = time.time() - t0

v2_accepted = sum(1 for sr in result_v2.step_results if not sr.rejected)
v2_total = result_v2.total_steps
v2_best_tm = max(
    (tm_score(sr.new_ca, target_ca) for sr in result_v2.step_results if not sr.rejected),
    default=tm_score(initial_ca, target_ca)
)
v2_best_rmsd = min(
    (compute_rmsd(sr.new_ca, target_ca) for sr in result_v2.step_results if not sr.rejected),
    default=compute_rmsd(initial_ca, target_ca)
)

# V3 final
r_v3 = ALL_RESULTS.get('phase4_final', [{}])[0]

print(f'\n{"="*60}')
print(f'{"V2 vs V3 COMPARISON":^60}')
print(f'{"="*60}')
print(f'{"Metric":<25} {"V2 Baseline":>15} {"V3 Composite":>15}')
print(f'{"-"*60}')
print(f'{"Accept rate":<25} {v2_accepted}/{v2_total} ({v2_accepted/max(1,v2_total)*100:.0f}%):>15 '
      f'{r_v3.get("accepted","?")}/{r_v3.get("total_steps","?")} ({r_v3.get("accept_rate",0)*100:.0f}%):>15')
print(f'{"Best TM-score":<25} {v2_best_tm:>15.4f} {r_v3.get("best_tm",0):>15.4f}')
print(f'{"Best RMSD (A)":<25} {v2_best_rmsd:>15.2f} {r_v3.get("best_rmsd",0):>15.2f}')
print(f'{"Wall time (s)":<25} {wall_v2:>15.0f} {r_v3.get("wall_s",0):>15.0f}')
print(f'{"="*60}')

## 11. Sonuclari Drive'a Kaydet

In [ ]:
# ════════════════════════════════════════════════════
#  SAVE TO DRIVE
# ════════════════════════════════════════════════════
import json
from datetime import datetime

save_dir = '/content/drive/MyDrive/ANM-openfold3/search_v3'
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Serialize
save_data = {}
for phase_name, results in ALL_RESULTS.items():
    save_data[phase_name] = []
    for r in results:
        r_copy = {k: v for k, v in r.items() if k != 'result'}
        save_data[phase_name].append(r_copy)

# Add V2 baseline
save_data['v2_baseline'] = {
    'accepted': v2_accepted, 'total_steps': v2_total,
    'best_tm': v2_best_tm, 'best_rmsd': v2_best_rmsd,
    'wall_s': wall_v2,
}

# Save JSON
save_path = os.path.join(save_dir, f'results_{timestamp}.json')
with open(save_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print(f'Results saved to {save_path}')

# Save summary CSV
all_rows = []
for phase_name, results in ALL_RESULTS.items():
    for r in results:
        all_rows.append({
            'phase': phase_name,
            'label': r.get('label', ''),
            'threshold': r.get('threshold', ''),
            'steps': r.get('total_steps', ''),
            'accepted': r.get('accepted', ''),
            'accept_rate': f"{r.get('accept_rate', 0)*100:.0f}%",
            'best_tm': r.get('best_tm', ''),
            'best_rmsd': r.get('best_rmsd', ''),
            'wall_s': r.get('wall_s', ''),
        })
csv_path = os.path.join(save_dir, f'summary_{timestamp}.csv')
pd.DataFrame(all_rows).to_csv(csv_path, index=False)
print(f'CSV saved to {csv_path}')

print('\nDone! Sonuclari Drive\'dan indir ve analiz et.')